# OCP7 - Réalisez une analyse de sentiments grâce au Deep Learning
# Notebook 2 - V1 du script python de classification de tweets : modèle simple

# Entraînement du modèle

In [3]:
# Example code to display the version of all imports
import pkg_resources

# List of package names
packages = ['pandas',
           'nltk',
           'scikit-learn',
           'joblib']

for package in packages:
    version = pkg_resources.get_distribution(package).version
    print(f"{package} == {version}")

pandas == 1.5.3
nltk == 3.8.1
scikit-learn == 1.2.2
joblib == 1.1.1


In [1]:
'''
This script uses the following python and packages versions :

python == 3.10.9 | packaged by Anaconda.

pandas == 1.5.3

nltk == 3.8.1

scikit-learn == 1.2.2

joblib == 1.1.1

'''

import pandas as pd 
import nltk
from nltk.corpus import stopwords
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from joblib import dump, load
import pickle
import warnings
warnings.filterwarnings("ignore")


def cleaning_stopwords(text):
    return " ".join([word for word in str(text).split() if word not in STOPWORDS])

def cleaning_punctuations(text):
    translator = str.maketrans('', '', punctuations_list)
    return text.translate(translator)

def cleaning_repeating_char(text):
    return re.sub(r'(.)\1+', r'\1', text)

def cleaning_email(data):
    return re.sub('@[^\s]+', ' ', data)

def cleaning_URLs(data):
    return re.sub('((www\.[^\s]+)|(https?://[^\s]+))',' ',data)

def cleaning_numbers(data):
    return re.sub('[0-9]+', '', data)

def stemming_on_text(data):
    text = [st.stem(word) for word in data]
    return data

def lemmatizer_on_text(data):
    text = [lm.lemmatize(word) for word in data]
    return data
## Data reading
data = pd.read_csv("data_raw/eda-twitter-sentiment-analysis-using-nn.csv", encoding = "ISO-8859-1", engine="python")
data.columns = ["label", "time", "date", "query", "username", "text"]

### Selecting text and label columns
data=data[['text','label']]

### Replacing label '4' with '1'
data['label'][data['label']==4]=1

### Separating positive and negative tweets
data_pos = data[data['label'] == 1]
data_neg = data[data['label'] == 0]

### Keeping only 25% of the data, to be able to run the program quickly at first
data_pos = data_pos.iloc[:int(20000)]
data_neg = data_neg.iloc[:int(20000)]

### Merging the two arrays to make an array with 20,000 positive tweets and 20,000 negative tweets
data = pd.concat([data_pos, data_neg])

### Converting uppercase to lowercase
data['text']=data['text'].str.lower()

### Removing English stopwords
stopwords_list = stopwords.words('english')

### Removing stop words
STOPWORDS = set(stopwords.words('english'))

data['text'] = data['text'].apply(lambda text: cleaning_stopwords(text))

### Removing punctuation
english_punctuations = string.punctuation
punctuations_list = english_punctuations
data['text']= data['text'].apply(lambda x: cleaning_punctuations(x))

### Removing repeated characters
data['text'] = data['text'].apply(lambda x: cleaning_repeating_char(x))

### Removing emails
data['text']= data['text'].apply(lambda x: cleaning_email(x))

### Removing URLs
data['text'] = data['text'].apply(lambda x: cleaning_URLs(x))

### Removing numbers
data['text'] = data['text'].apply(lambda x: cleaning_numbers(x))

### Tokenization of tweets
tokenizer = nltk.tokenize.RegexpTokenizer(r'\w+')
data['text'] = data['text'].apply(tokenizer.tokenize)

### Stemming
st = nltk.PorterStemmer()
data['text']= data['text'].apply(lambda x: stemming_on_text(x))

### Lemmatization
lm = nltk.WordNetLemmatizer()
data['text'] = data['text'].apply(lambda x: lemmatizer_on_text(x))

## Bag of Words Approach: Tf-idf

# Formatting needed for Bow approach
data['text_string'] = data['text'].apply(lambda x: ' '.join(x))

# Define X and y
X = data['text_string']
y = data['label']

# Instantiate count_vectorizer
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.95, min_df=1)

# Fit and transform the data
X = vectorizer.fit_transform(data['text_string'])

# Separate training and test data (70% - 30%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2)

# Make the model with the specified regularization parameter
log_reg = LogisticRegression(C = 0.0001)

# Train on the training data
log_reg.fit(X_train, y_train)

# Make predictions from the test data
y_pred = log_reg.predict_proba(X_test)[:, 1]

### Performance Evaluation

#### ROC Area Under Curve

# Calculate the ROC AUC score
roc_auc = roc_auc_score(y_test, y_pred)

# Display the rounded result
print(f"ROC AUC score: {round(roc_auc,2)}")

# Save the model
dump(log_reg, 'baseline.joblib')

# Save the vectorizer
with open('vectorizer.pkl', 'wb') as file:
    pickle.dump(vectorizer, file)

ROC AUC score: 0.77
